## Imports

In [ ]:
import itertools

import numpy as np

import torch

import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Quantum Gates and Utilities

In [ ]:
dtype = torch.complex64
print(f"dtype: {dtype}")

device = "cpu"
print(f"device: {device}")

In [ ]:
i2 = torch.eye(2, dtype=dtype, device=device)
print(f"I2: {i2}")

px = torch.tensor([[0, 1], [1, 0]], dtype=dtype, device=device)
print(f"Pauli X: {px}")

py = torch.tensor([[0, -1j], [1j, 0]], dtype=dtype, device=device)
print(f"Pauli Y: {py}")

pz = torch.tensor([[1, 0], [0, -1]], dtype=dtype, device=device)
print(f"Pauli Z: {pz}")

pauli_1q = {0: i2, 1: px, 2: py, 3: pz}

for key, value in pauli_1q.items():
    print(f"{key}: {value}")

pauli_labels = {0: 'i2', 1: 'px', 2: 'py', 3: 'pz'}

for key, value in pauli_labels.items():
    print(f"{key}: {value}", end=",    ")

In [ ]:
def rotation_gate(pauli_gate, theta):
    
    return torch.cos(theta / 2)*i2 - 1j * torch.sin(theta / 2) * pauli_gate

In [ ]:
theta = torch.tensor(torch.pi / 2)
print(f"theta: {theta}")

rx = rotation_gate(px, theta)
print(f"rx: {rx}")

In [ ]:
print(f"rx unitarity check: {torch.conj(rx).T @ rx}")

In [ ]:
# kronecker product of a list of matrices

def kron_list(matrices):
    
    result = matrices[0]

    for matrix in matrices[1:]:
        result = torch.kron(result, matrix)

    return result

In [ ]:
result = kron_list([i2, i2, i2])
print(f"result: {result}")

In [ ]:
# embed single-qubit gate into n-qubit space

def embed_1q_gate(gate_2x2, target, n_qubits):
    
    matrices = [i2]*n_qubits
    
    matrices[target] = gate_2x2
    
    return kron_list(matrices)

In [ ]:
result = embed_1q_gate(px, 1, 2)
print(f"result: {result}")

In [ ]:
p0 = torch.tensor([[1, 0],
                   [0, 0]], dtype=dtype, device=device)

p1 = torch.tensor([[0, 0],
                   [0, 1]], dtype=dtype, device=device)

# build CNOT gate in n-qubit space

def cnot_gate(control, target, n_qubits):
    
    # will store matrices for the case when control = 0
    mats_0 = []
    
    # will store matrices for the case when control = 1
    mats_1 = []

    # loop over each qubit position
    for i in range(n_qubits):
        
        # if current qubit is the control qubit
        if i == control:
            
            # for control = 0 branch -> use projector | 0 >< 0 |
            mats_0.append(p0)
            
            # for control = 1 branch -> use projector | 1 >< 1 |
            mats_1.append(p1)
        
        # if current qubit is the target qubit
        elif i == target:
            
            # if control = 0 -> do nothing -> Identity
            mats_0.append(i2)
            
            # if control = 1 -> apply X (flip target qubit)
            mats_1.append(px)
        
        # for all other qubits (neither control nor target)
        else:
            
            # identity (no change)
            mats_0.append(i2)
            
            # identity (no change)
            mats_1.append(i2)

    return kron_list(mats_0) + kron_list(mats_1)

In [ ]:
cnot_4x4 = cnot_gate(0, 1, 2)
print(f"cnot_4x4: {cnot_4x4}")

In [ ]:
n_qubits = 2

for indices in itertools.product(range(4), repeat=n_qubits):
    print(indices)

In [ ]:
def get_pauli_strings(n_qubits, max_weight=None):
    
    # will store results as tuples: (matrix, label, weight)
    strings = []
    
    # generate all combinations of length n_qubits
    # each element is in {0,1,2,3} -> {I, X, Y, Z}
    # example for n=2: (0,1), (2,3), etc.
    for indices in itertools.product(range(4), repeat=n_qubits):
        
        # Skip the all-identity case
        if indices == (0,) * n_qubits:
            continue
        
        # count how many non-identity operators are present
        # example: (0,1,3) -> weight = 2 (X and Z)
        weight = sum(1 for idx in indices if idx != 0)
        
        # if user specified a max weight, skip larger ones
        if max_weight is not None and weight > max_weight:
            continue
    
        # convert each index to its corresponding 2x2 matrix
        # (1,3) -> [X, Z]
        matrices = [pauli_1q[idx] for idx in indices]
        
        # compute full n-qubit operator using tensor product
        matrix = kron_list(matrices)
        
        # convert indices to string label
        # (1,3) -> "XZ"
        label = ''.join(pauli_labels[idx] for idx in indices)
        
        strings.append((matrix, label, weight))
        
    return strings

In [ ]:
ops = get_pauli_strings(n_qubits=2, max_weight=1)

for mat, label, wt in ops:
    print(label, wt, mat)

```
q0: ─ Rx ─ Ry ─ Rz ───■────────────────
                      │
q1: ─ Rx ─ Ry ─ Rz ───X ── Rz ───■─────
                                 │
q2: ─ Rx ─ Ry ─ Rz ──────────────X ── Rz
```

In [ ]:
def build_generator_unitary(angles, n_qubits):
    
    dim = 2 ** n_qubits
    
    U = torch.eye(dim, dtype=dtype, device=device)
    
    n_sq = n_qubits * 3
    
    # single-qubit rotations
    for q in range(n_qubits):
        
        for k, pauli_gate in enumerate([px, py, pz]):
            
            gate = embed_1q_gate(rotation_gate(pauli_gate, angles[q*3 + k]), q, n_qubits)
            
            U = gate @ U
    
    # entangling layer: CNOT between adjacent qubits + rz
    if n_qubits > 1:
        
        for q in range(n_qubits - 1):
            
            U = cnot_gate(q, q + 1, n_qubits) @ U
            
            if n_sq + q < len(angles):
                
                rz = embed_1q_gate(rotation_gate(pz, angles[n_sq + q]), q + 1, n_qubits)
                
                U = rz @ U
    
    return U

In [ ]:
n_qubits = 2
num_params = 3 * n_qubits + (n_qubits - 1)
angles = torch.randn(num_params, dtype=torch.float32)
U = build_generator_unitary(angles, n_qubits)
print(f"U: {U}")

In [ ]:
def zero_state(n_qubits):
    
    # state = torch.zeros((2 ** n_qubits, 1), dtype=dtype, device=device)
    
    # state[0, 0] = 1.0
    
    # return state
    
    N = 2 ** n_qubits
    state = torch.zeros(N, dtype=dtype, device=device)
    
    return torch.cat([
        torch.tensor([1.0], dtype=dtype, device=device),
        state[1:]
    ])

In [ ]:
n3_zero_state = zero_state(3)
print(f"3 qubit zero state: {n3_zero_state}")

In [ ]:
def extract_bloch(state, pauli_strings):
    
    # convert to density matrix if pure state
    # if state.ndim == 1:
    #     rho = state.unsqueeze(1) @ state.conj().unsqueeze(0)
    
    # else:
    #     rho = state
    
    if state.ndim == 1:
        # shape: (N,) -> pure state
        rho = state.unsqueeze(1) @ state.conj().unsqueeze(0)

    elif state.ndim == 2 and state.shape[1] == 1:
        # shape: (N,1) -> column vector (pure state)
        rho = state @ state.conj().T

    elif state.ndim == 2 and state.shape[0] == state.shape[1]:
        # shape: (N,N) -> already density matrix
        rho = state

    else:
        raise ValueError(f"Invalid state shape: {state.shape}")

    values = []
    
    for matrix, _, _ in pauli_strings:
        val = torch.real(torch.trace(rho @ matrix))
        values.append(val)

    return torch.stack(values)

In [ ]:
def fidelity_pure(gen_angles, real_state, n_qubits):
    
    # build parameterized unitary U(theta) from generator
    U = build_generator_unitary(gen_angles, n_qubits)
    
    # apply U to zero state -> generated (fake) quantum state
    fake = U @ zero_state(n_qubits)
    
    # compute inner product < real | fake >
    overlap = real_state.conj().T @ fake
    
    # return fidelity = |< real | fake >|^2
    return torch.abs(overlap) ** 2

In [ ]:
def create_random_target(n_qubits):
    
    # total number of parameters
    # 3 per qubit (rx, ry, rz) + (n - 1) entangling params
    n_params = n_qubits * 3 + max(0, n_qubits - 1)
    
    # random angles in range ~[-2pi, 2pi]
    angles = torch.randn(n_params) * 2 * torch.pi
    
    # build random unitary using same ansatz
    U = build_generator_unitary(angles, n_qubits)
    
    # Apply to zero state -> random quantum state
    return U @ zero_state(n_qubits)

In [ ]:
n_qubits = 2

# create a random "real" target state
real_state = create_random_target(n_qubits)

# initialize generator parameters
n_params = 3*n_qubits + (n_qubits - 1)
gen_angles = torch.randn(n_params, requires_grad=True)

F = fidelity_pure(gen_angles, real_state, n_qubits)

print(f"Fidelity: {F.item()}")

## BlochWGAN Pure

In [ ]:
class BlochWGANPure:
    def __init__(self, real_state, n_qubits, pauli_strings, lr):
        
        # target state
        self.real = real_state

        self.n = n_qubits
        self.paulis = pauli_strings
        self.lr = lr

        # dimension of Bloch vector
        self.d = len(pauli_strings)

        # real Bloch vector
        self.r_target = extract_bloch(real_state, pauli_strings)

        # generator parameters
        n_params = n_qubits * 3 + max(0, n_qubits - 1)
        self.gen = torch.randn(n_params, requires_grad=True)

        # discriminator weights (linear)
        self.w = torch.randn(self.d, requires_grad=True)

    def forward(self):
        
        # generate fake state
        U = build_generator_unitary(self.gen, self.n)
        fake = U @ zero_state(self.n)

        # Bloch vector of fake state
        r_fake = extract_bloch(fake, self.paulis)

        # Wasserstein loss (critic)
        # maximize w(real - fake) -> minimize negative
        loss_disc = -torch.dot(self.w, self.r_target - r_fake)

        # generator tries to minimize -w r_fake
        loss_gen = -torch.dot(self.w.detach(), r_fake)

        return loss_disc, loss_gen, fake, r_fake

In [ ]:
def trainBlochWGANPure(
    n_qubits: int = 8,         # number of qubits in the system
    epochs: int = 400,         # number of training iterations
    lr: float = 0.1,           # learning rate for both generator and discriminator
    max_weight: int = 2,       # max Pauli weight (limits observable complexity)
    seed: int | None = None,   # random seed for reproducibility
):
    
    if seed is not None:
        # fix randomness for reproducibility
        torch.manual_seed(seed)

    # generate a random target quantum state
    real_state = create_random_target(n_qubits)
    
    # generate subset of Pauli operators used to form Bloch vector
    paulis = get_pauli_strings(n_qubits, max_weight=max_weight)

    # initialize Bloch-WGAN model
    model = BlochWGANPure(real_state=real_state, n_qubits=n_qubits, pauli_strings=paulis, lr=lr)
    
    # optimizer for generator parameters
    opt_gen = torch.optim.SGD([model.gen], lr=lr)
    
    # optimizer for discriminator weights
    opt_disc = torch.optim.SGD([model.w], lr=lr)

    # lists to track training progress
    fidelities, losses = [], []

    for epoch in range(epochs):
        
        # ----------- discriminator step -----------

        # clear previous gradients of discriminator
        opt_disc.zero_grad()
        
        # compute:
        # - discriminator loss
        # - generator loss
        # - fake quantum state
        # - fake Bloch vector
        loss_disc, loss_gen, fake, r_fake = model.forward()
        
        # compute gradients w.r.t discriminator weights
        loss_disc.backward(retain_graph=True)
        
        # update discriminator weights
        opt_disc.step()

        # enforce Lipschitz constraint (discriminator weights)
        with torch.no_grad():
            norm = torch.norm(model.w)
            if norm > 1:
                # project weights onto unit ball
                model.w /= norm

        # ----------- generator step -----------
        
        # clear previous gradients of generator
        opt_gen.zero_grad()
        
        # recompute forward pass (updated discriminator)
        loss_disc, loss_gen, fake, r_fake = model.forward()
        
        # compute gradients w.r.t generator parameters theta
        loss_gen.backward()
        
        # update thetas
        opt_gen.step()

        # compute fidelity between generated and real quantum state
        fid = fidelity_pure(model.gen, model.real, model.n)
        
        # store fidelity value
        fidelities.append(fid.item())
        
        # store generator loss
        losses.append(loss_gen.item())

        print(f"Epoch {epoch+1:4d}/{epochs} | Loss: {loss_gen.item():+.6f} | Fidelity: {fid.item():.6f}")

    print(f"Training complete | Final fidelity: {fidelities[-1]:.6f}")
    
    # return training history
    return {"fidelities": fidelities, "losses": losses}

## BlochWGAN Mixed

In [ ]:
def get_density_matrix(state):
    if state.ndim == 1:
        return state.unsqueeze(1) @ state.conj().unsqueeze(0)
    elif state.ndim == 2 and state.shape[1] == 1:
        return state @ state.conj().T
    return state

In [ ]:
def random_mixed_state_density_matrix(n_qubits, prob_real):
    dim = 2 ** n_qubits
    density_matrix = torch.zeros((dim, dim), dtype=dtype, device=device)

    for p in prob_real:
        state = create_random_target(n_qubits)
        density_matrix += p * get_density_matrix(state)

    return density_matrix

In [ ]:
def fidelity_mixed(rho_fake, rho_real):
    # F(\rho, \sigma) = \left( 
    # \mathrm{Tr} \sqrt{\sqrt{\rho}\,\sigma\,
    # \sqrt{\rho}} \right)^{2},
    eigvals = torch.linalg.eigvalsh(rho_real)
    eigvals = torch.clamp(eigvals, min=0.0)
    sqrt_real = torch.zeros_like(rho_real)

    # compute $\rho_real$ via eigendecomposition
    eigvals_full, eigvecs = torch.linalg.eigh(rho_real)
    eigvals_full = torch.clamp(eigvals_full, min=0.0)
    sqrt_eigvals = torch.sqrt(eigvals_full)
    sqrt_real = eigvecs @ torch.diag(sqrt_eigvals.to(dtype)) @ eigvecs.conj().T

    M = sqrt_real @ rho_fake @ sqrt_real
    eigvals_M = torch.linalg.eigvalsh(M)
    eigvals_M = torch.clamp(eigvals_M.real, min=0.0)
    
    fid = torch.sum(torch.sqrt(eigvals_M)) ** 2

    return torch.clamp(fid, max=1.0)

In [ ]:

class BlochWGANMixed:
    def __init__(self, rho_real, n_qubits, pauli_strings, n_components=2):
        self.rho_real = rho_real
        self.n = n_qubits
        self.paulis = pauli_strings
        self.d = len(pauli_strings)
        self.n_components = n_components
        self.lamb = 0.1

        # Bloch vector of real mixed state
        self.r_target = extract_bloch(rho_real, pauli_strings)

        # generator: one set of circuit angles per pure state component
        n_params = n_qubits * 3 + max(0, n_qubits - 1)
        self.gen_angles = [torch.randn(n_params, requires_grad=True) for _ in range(n_components)]

        # learnable mixing logits -> softmax gives probabilities
        self.mix_logits = torch.zeros(n_components, requires_grad=True)

        # discriminator weights
        self.w = torch.randn(self.d, requires_grad=True)

    @property
    def mix_probs(self):
        return torch.softmax(self.mix_logits, dim=0)

    def gen_params(self):
        return self.gen_angles + [self.mix_logits]

    def fake_density_matrix(self):
        dim = 2 ** self.n
        probs = self.mix_probs
        rho = torch.zeros((dim, dim), dtype=dtype, device=device)

        for i in range(self.n_components):
            U = build_generator_unitary(self.gen_angles[i], self.n)
            state = U @ zero_state(self.n)
            rho = rho + probs[i] * get_density_matrix(state)

        return rho

    def forward(self):
        rho_fake = self.fake_density_matrix()
        r_fake = extract_bloch(rho_fake, self.paulis)

        loss_disc = -torch.dot(self.w, self.r_target - r_fake)
        loss_gen = -torch.dot(self.w.detach(), r_fake)

        return loss_disc, loss_gen, rho_fake, r_fake

    def reset_angles(self):
        with torch.no_grad():
            for angles in self.gen_angles:
                angles.uniform_(0, 1)

In [ ]:
def trainBlochWGANMixed(
    n_qubits: int = 3,
    epochs: int = 400,
    lr: float = 0.1,
    prob_real: list = [0.2, 0.8],
    max_weight: int = 2,
    seed: int | None = None
):
    if seed is not None:
        # fix randomness for reproducibility
        torch.manual_seed(seed)
    
    # generate a random target density matrix of mixed state
    rho_real = random_mixed_state_density_matrix(n_qubits, prob_real)
    
    # generate subset of Pauli operators used to form Bloch vector
    paulis = get_pauli_strings(n_qubits, max_weight=max_weight)

    # initialize Bloch-WGAN model
    model = BlochWGANMixed(
        rho_real=rho_real,
        n_qubits=n_qubits,
        pauli_strings=paulis,
        n_components=len(prob_real)
    )

    # optimizer for generator parameters
    opt_gen = torch.optim.SGD(model.gen_params(), lr=lr)
    
    # optimizer for discriminator weights
    opt_disc = torch.optim.SGD([model.w], lr=lr)
    
    # lists to track training progress
    fidelities, losses = [], []
    
    for epoch in range(epochs):
        
        # ----------- discriminator step -----------

        # clear previous gradients of discriminator
        opt_disc.zero_grad()
        
        # compute:
        # - discriminator loss
        # - generator loss
        # - fake mixed state density matrix
        # - fake mixed state Bloch vector
        loss_disc, loss_gen, rho_fake, r_fake = model.forward()
        
        # compute gradients w.r.t discriminator weights
        loss_disc.backward(retain_graph=True)
        
        # update discriminator weights
        opt_disc.step()

        # enforce Lipschitz constraint (discriminator weights)
        with torch.no_grad():
            norm = torch.norm(model.w)
            if norm > 1:
                # project weights onto unit ball
                model.w /= norm

        # ----------- generator step -----------

        # clear previous gradients of generator
        opt_gen.zero_grad()
        
        # recompute forward pass (updated discriminator)
        loss_disc, loss_gen, rho_fake, r_fake = model.forward()
        
        # compute gradients w.r.t generator parameters theta
        loss_gen.backward()
        
        # update thetas
        opt_gen.step()

        # compute fidelity between generated and real quantum state
        fid = fidelity_mixed(rho_fake.detach(), model.rho_real)
        
        probs = model.mix_probs.detach()
        
        # store fidelity value
        fidelities.append(fid.item())
        
        # store generator loss
        losses.append(loss_gen.item())

        print(f"Epoch {epoch+1:4d}/{epochs} | Loss: {loss_gen.item():+.6f} | Fidelity: {fid.item():.6f}")
        
    print(f"Training complete | Final fidelity: {fidelities[-1]:.6f}")
    
    # return training history
    return {"fidelities": fidelities, "losses": losses}

In [ ]:
class BlochWGANMixedWithReg:
    def __init__(self, rho_real, n_qubits, pauli_strings, n_components=2):
        self.rho_real = rho_real
        self.n = n_qubits
        self.paulis = pauli_strings
        self.d = len(pauli_strings)
        self.n_components = n_components
        self.lamb = 0.1

        # Bloch vector of real mixed state
        self.r_target = extract_bloch(rho_real, pauli_strings)

        # generator: one set of circuit angles per pure state component
        n_params = n_qubits * 3 + max(0, n_qubits - 1)
        self.gen_angles = [torch.randn(n_params, requires_grad=True) for _ in range(n_components)]

        # learnable mixing logits -> softmax gives probabilities
        self.mix_logits = torch.zeros(n_components, requires_grad=True)

        # discriminator weights
        self.w = torch.randn(self.d, requires_grad=True)

    @property
    def mix_probs(self):
        return torch.softmax(self.mix_logits, dim=0)

    def gen_params(self):
        return self.gen_angles + [self.mix_logits]

    def fake_density_matrix(self):
        dim = 2 ** self.n
        probs = self.mix_probs
        rho = torch.zeros((dim, dim), dtype=dtype, device=device)

        for i in range(self.n_components):
            U = build_generator_unitary(self.gen_angles[i], self.n)
            state = U @ zero_state(self.n)
            rho = rho + probs[i] * get_density_matrix(state)

        return rho
    
    def forward(self):
        rho_fake = self.fake_density_matrix()
        r_fake = extract_bloch(rho_fake, self.paulis)

        diff = self.r_target - r_fake
        loss_disc = -torch.dot(self.w, diff)

        # Bloch-space entropic regulariser
        reg = self.lamb * torch.sum(self.w ** 2)
        loss_disc = loss_disc + reg

        loss_gen = -torch.dot(self.w.detach(), r_fake)

        return loss_disc, loss_gen, rho_fake, r_fake

    def reset_angles(self):
        with torch.no_grad():
            for angles in self.gen_angles:
                angles.uniform_(0, 1)

In [ ]:
def trainBlochWGANMixedWithReg(
    n_qubits: int = 3,
    epochs: int = 400,
    lr: float = 0.1,
    prob_real: list = [0.2, 0.8],
    max_weight: int = 2,
    seed: int | None = None
):
    if seed is not None:
        # fix randomness for reproducibility
        torch.manual_seed(seed)
    
    # generate a random target density matrix of mixed state
    rho_real = random_mixed_state_density_matrix(n_qubits, prob_real)
    
    # generate subset of Pauli operators used to form Bloch vector
    paulis = get_pauli_strings(n_qubits, max_weight=max_weight)

    # initialize Bloch-WGAN model
    model = BlochWGANMixedWithReg(
        rho_real=rho_real,
        n_qubits=n_qubits,
        pauli_strings=paulis,
        n_components=len(prob_real)
    )

    # optimizer for generator parameters
    opt_gen = torch.optim.SGD(model.gen_params(), lr=lr)
    
    # optimizer for discriminator weights
    opt_disc = torch.optim.SGD([model.w], lr=lr)
    
    # lists to track training progress
    fidelities, losses = [], []
    
    for epoch in range(epochs):
        
        # ----------- discriminator step -----------

        # clear previous gradients of discriminator
        opt_disc.zero_grad()
        
        # compute:
        # - discriminator loss
        # - generator loss
        # - fake mixed state density matrix
        # - fake mixed state Bloch vector
        loss_disc, loss_gen, rho_fake, r_fake = model.forward()
        
        # compute gradients w.r.t discriminator weights
        loss_disc.backward(retain_graph=True)
        
        # update discriminator weights
        opt_disc.step()

        # enforce Lipschitz constraint (discriminator weights)
        with torch.no_grad():
            norm = torch.norm(model.w)
            if norm > 1:
                # project weights onto unit ball
                model.w /= norm

        # ----------- generator step -----------

        # clear previous gradients of generator
        opt_gen.zero_grad()
        
        # recompute forward pass (updated discriminator)
        loss_disc, loss_gen, rho_fake, r_fake = model.forward()
        
        # compute gradients w.r.t generator parameters theta
        loss_gen.backward()
        
        # update thetas
        opt_gen.step()

        # compute fidelity between generated and real quantum state
        fid = fidelity_mixed(rho_fake.detach(), model.rho_real)
        
        probs = model.mix_probs.detach()
        
        # store fidelity value
        fidelities.append(fid.item())
        
        # store generator loss
        losses.append(loss_gen.item())

        print(f"Epoch {epoch+1:4d}/{epochs} | Loss: {loss_gen.item():+.6f} | Fidelity: {fid.item():.6f}")
        
    print(f"Training complete | Final fidelity: {fidelities[-1]:.6f}")
    
    # return training history
    return {"fidelities": fidelities, "losses": losses}

## Experiments

In [ ]:
def plot_mean_with_variance(metrics: dict, meta: dict = None, colors: dict = None):
    # number of metrics
    num_plots = len(metrics)

    fig = make_subplots(
        rows=1,                                # single row layout
        cols=num_plots,                        # one subplot per metric
        # subplot_titles=list(metrics.keys()),   # titles = metric names
        horizontal_spacing=0.08                # spacing between plots
    )

    # iterate over each metric
    # runs = list of lists (each inner list = one run)
    for i, (name, runs) in enumerate(metrics.items(), start=1):

        # convert runs into NumPy array
        data = np.array(runs)

        # mean across runs (per epoch)
        mean = data.mean(axis=0)
        
        # standard deviation across runs
        std = data.std(axis=0)

        # x-axis = epochs
        x = np.arange(len(mean))

        # upper bound of variance band
        upper = mean + std
        
        # lower bound of variance band
        lower = mean - std
        
        if colors is not None:
            color = colors.get(name, "rgba(0,200,0,1)")
        else:
            color = "rgba(0,200,0,1)"
            
        if "rgba" in color:
            fill_color = color.replace("1)", "0.2)")
        else:
            fill_color = "rgba(0,200,0,0.2)"

        fig.add_trace(
            go.Scatter(
                x=np.concatenate([x, x[::-1]]),           # create closed loop for shading (forward + reverse)
                y=np.concatenate([upper, lower[::-1]]),   # upper curve + reversed lower curve
                fill='toself',                            # fill area between curves
                fillcolor=fill_color,                     # transparent green shading
                line=dict(color='rgba(255,255,255,0)'),   # invisible boundary
                hoverinfo="skip",                         # disable hover for shaded region
                showlegend=False                          # hide legend
            ),
            row=1,
            col=i
        )

        fig.add_trace(
            go.Scatter(
                x=x,                                # epochs
                y=mean,                             # mean curve
                mode='lines',                       # line plot
                line=dict(color=color, width=3),  # bold black line
                name=f"{name} Mean"                 # label (not shown since legend disabled)
            ),
            row=1,
            col=i
        )

        # label x-axis
        fig.update_xaxes(title_text="Epoch", row=1, col=i)
        
        # label y-axis
        fig.update_yaxes(title_text=name, row=1, col=i)
        
    # title = "Mean Training Curves with Variance"
    title = ""
    if meta:
        meta_str = " | ".join([f"{k}={v}" for k, v in meta.items()])
        title += f"{meta_str}"

    fig.update_layout(
        height=400,                                 # figure height
        width=500 * num_plots,                      # dynamic width
        # title=title, # title
        template="plotly_white",                    # clean theme
        showlegend=False                            # hide legend
    )

    # fig.write_image(f"../images/{title}.png")
    # pio.write_image(fig, f"../images/{title}.png",scale=6, width=1080, height=1080)
    pio.write_image(fig, f"../images/{title}.png", scale=4, width=1600, height=900)
    
    # return Plotly figure
    return fig

In [ ]:
def multiple_exp(num_runs, train, label="", color="rgba(0,200,0,1)"):
    fidelities_runs = []
    losses_runs = []

    for i in range(num_runs):
        print(f"Run {i+1}/{num_runs}")

        result = train()

        fidelities_runs.append(result["fidelities"])
        # losses_runs.append(result["losses"])
    
    fig = plot_mean_with_variance(
        {
            "Fidelity": fidelities_runs,
            # "Generator Loss": losses_runs
        },
        meta={"Experiment": label},
        colors={
            "Fidelity": color,
            "Loss": color
        }
    )
    
    fig.show()

In [ ]:
colors = [
    "rgba(0,128,255,1)",    # blue
    "rgba(255,0,0,1)",      # red
    "rgba(0,200,0,1)",      # green
    "rgba(255,165,0,1)",    # orange
    "rgba(128,0,128,1)",    # purple
    "rgba(0,255,255,1)",    # cyan
    "rgba(255,20,147,1)",   # deep pink
    "rgba(255,215,0,1)",    # gold
    "rgba(0,128,128,1)",    # teal
    "rgba(139,69,19,1)",    # brown
    "rgba(75,0,130,1)",     # indigo
]

### Pure State Experiments: `1-4 qubit`

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANPure(
#         n_qubits=1,
#         epochs=150,
#         lr=0.1
#     ),
#     label="1Q Pure",
#     color=colors[0]
# )

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANPure(
#         n_qubits=2,
#         epochs=150,
#         lr=0.1
#     ),
#     label="2Q Pure",
#     color=colors[1]
# )

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANPure(
#         n_qubits=3,
#         epochs=150,
#         lr=0.1,
#         max_weight=2
#     ),
#     label="3Q Pure",
#     color=colors[2]
# )

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANPure(
#         n_qubits=4,
#         epochs=300,
#         lr=0.1,
#         max_weight=2
#     ),
#     label="4Q Pure",
#     color=colors[3]
# )

### Mixed State Experiments: `1-3` qubit

In [ ]:
# multiple_exp(
#     50, 
#     lambda: trainBlochWGANMixed(
#         n_qubits=1,
#         epochs=150,
#         lr=0.1
#     ),
#     label="1Q Mixed",
#     color=colors[5]
# )

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANMixed(
#         n_qubits=2, 
#         epochs=600,
#         lr=0.1
#     ),
#     label="2Q Mixed",
#     color=colors[6]
# )

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANMixed(
#         epochs=800
#     ),
#     label="3Q Mixed",
#     color=colors[7]
# )

### Mixed State With Reg Experiments: `1-3` qubit

In [ ]:
# multiple_exp(
#     50, 
#     lambda: trainBlochWGANMixedWithReg(
#         n_qubits=1,
#         epochs=150,
#         lr=0.1
#     ),
#     label="1Q Mixed With Reg",
#     color=colors[8]
# )

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANMixedWithReg(
#         n_qubits=2, 
#         epochs=600,
#         lr=0.1
#     ),
#     label="2Q Mixed With Reg",
#     color=colors[9]
# )

In [ ]:
# multiple_exp(
#     50,
#     lambda: trainBlochWGANMixedWithReg(
#         epochs=800
#     ),
#     label="3Q Mixed With Reg",
#     color=colors[10]
# )

### Pure State Experiment: `8`-qubit

In [ ]:
multiple_exp(
    50,
    lambda: trainBlochWGANPure(),
    label="8Q Pure",
    color=colors[4]
)

In [ ]:
import time

def time_config(train_fn, n_runs=5, warmup=1):
    # discard first run (allocator / cache warmup)
    for _ in range(warmup):
        train_fn()

    times = []

    for _ in range(n_runs):
        t0 = time.perf_counter()
        train_fn()
        times.append(time.perf_counter() - t0)
    return float(np.mean(times)), float(np.std(times))

# deterministic, reportable timings
torch.set_num_threads(1)

timing_configs = {
    "1q-pure": lambda: trainBlochWGANPure(n_qubits=1, epochs=150, lr=0.1, max_weight=1),
    "2q-pure": lambda: trainBlochWGANPure(n_qubits=2, epochs=150, lr=0.1, max_weight=2),
    "3q-pure": lambda: trainBlochWGANPure(n_qubits=3, epochs=150, lr=0.1, max_weight=2),
    "4q-pure": lambda: trainBlochWGANPure(n_qubits=4, epochs=300, lr=0.1, max_weight=2),
    "8q-pure": lambda: trainBlochWGANPure(n_qubits=8, epochs=400, lr=0.1, max_weight=2),
    "1q-mixed": lambda: trainBlochWGANMixed(n_qubits=1, epochs=150, lr=0.1),
    "2q-mixed": lambda: trainBlochWGANMixed(n_qubits=2, epochs=600, lr=0.1),
    "3q-mixed": lambda: trainBlochWGANMixed(n_qubits=3, epochs=800, lr=0.1),
    "1q-mixed-reg": lambda: trainBlochWGANMixedWithReg(n_qubits=1, epochs=150, lr=0.1),
    "2q-mixed-reg": lambda: trainBlochWGANMixedWithReg(n_qubits=2, epochs=600, lr=0.1),
    "3q-mixed-reg": lambda: trainBlochWGANMixedWithReg(n_qubits=3, epochs=800, lr=0.1),
}

for name, fn in timing_configs.items():
    mean, std = time_config(fn, n_runs=5)
    print(f"{name:10s}  {mean:8.2f} +/- {std:.2f} s")